In [ ]:
#!/usr/bin/env python3
"""
train.py
────────
Script huấn luyện mô hình PyTorch Offline dùng cho cơ chế DAgger (Imitation Learning).
Tùy biến để chạy trực tiếp trên Google Colab với các file nằm cùng cấp thư mục.

Cải tiến so với bản gốc:
  1. Chia Train/Val theo BLOCK liên tục theo thời gian (giảm data leakage do LiDAR
     frame liền kề gần như giống hệt nhau) thay vì random theo từng dòng.
     Nếu CSV có cột 'episode', sẽ chia theo episode (chuẩn nhất).
  2. Chuẩn hóa target (linear_v, angular_z) để MSE không bị linear_v áp đảo.
  3. Log riêng MAE cho từng output (velocity vs steering) mỗi epoch.
  4. Early stopping dựa trên val loss, tránh train dư epoch.
  5. weight_decay + LR scheduler (ReduceLROnPlateau).
  6. Seed cố định để reproduce được kết quả.
  7. Đọc CSV bằng numpy thay vì vòng lặp Python thuần -> nhanh hơn nhiều với 29k dòng.
"""

import os
import csv
import json
import numpy as np

# PyTorch Imports
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader, Subset
except ImportError:
    raise ImportError("Thiếu thư viện PyTorch! Hãy cài đặt bằng lệnh: pip install torch torchvision")


SEED = 42


def set_seed(seed=SEED):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# --- 1. Định nghĩa Mạng Neural (Đồng bộ với dagger_inference.py) ---
class DAggerMLP(nn.Module):
    def __init__(self, input_dim=60, output_dim=2, dropout=0.1):
        super(DAggerMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.network(x)


# --- 2. Dataset Custom cho DAgger ---
class DAggerDataset(Dataset):
    """
    Đọc CSV với input_dim cột LiDAR đầu + 2 cột cuối (linear_v, angular_z).
    Nếu CSV có cột tên 'episode' (hoặc 'run_id'/'run'), sẽ lưu lại để hỗ trợ
    chia train/val theo episode thay vì random theo dòng.

    Target được chuẩn hóa bằng (x - mean) / std, các giá trị mean/std được
    lưu lại trong self.target_mean / self.target_std để dùng lúc inference
    (phải nhân ngược lại khi suy luận thực tế trên xe).
    """

    LIDAR_MAX_RANGE = 10.0  # m — chỉnh lại nếu range_max của sensor/sim khác

    def __init__(self, csv_path, input_dim=60):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Không tìm thấy file dataset tại: {csv_path}. Hãy tải file lên Colab!")

        self.input_dim = input_dim
        expected_cols = input_dim + 2  # + speed, steering_angle (+ episode nếu có, xử lý riêng bên dưới)

        with open(csv_path, 'r', newline='') as f:
            reader = csv.reader(f)
            header = next(reader)

            episode_col = None
            for cand in ('episode', 'run_id', 'run'):
                if cand in header:
                    episode_col = header.index(cand)
                    break

            # Số cột đúng của header (LiDAR + speed + steering [+ episode])
            expected_cols_header = len(header)

            good_rows = []
            n_bad = 0
            for line_num, row in enumerate(reader, start=2):  # dòng 1 là header
                if len(row) != expected_cols_header:
                    n_bad += 1
                    continue
                good_rows.append(row)

        if n_bad > 0:
            print(f"⚠️  Bỏ qua {n_bad} dòng CSV bị lỗi (sai số cột, thường do ghi file dở dang) "
                  f"trên tổng {n_bad + len(good_rows)} dòng.")

        if len(good_rows) == 0:
            raise ValueError(f"Không có dòng dữ liệu hợp lệ nào trong {csv_path}!")

        raw = np.array(good_rows, dtype=str)

        # Kiểm tra input_dim có khớp với số cột LiDAR thực tế trong file không
        actual_lidar_cols = expected_cols_header - (2 + (1 if episode_col is not None else 0))
        if actual_lidar_cols != input_dim:
            raise ValueError(
                f"input_dim={input_dim} KHÔNG khớp với số cột LiDAR thực tế trong CSV ({actual_lidar_cols}). "
                f"Nếu dùng sai input_dim, các cột target (speed, steering_angle) sẽ bị đọc lệch sang cột LiDAR khác "
                f"-> model học sai hoàn toàn. Hãy truyền đúng --input_dim {actual_lidar_cols}."
            )

        self.inputs = raw[:, :input_dim].astype(np.float32)
        self.targets = raw[:, input_dim:input_dim + 2].astype(np.float32)

        if episode_col is not None:
            self.episodes = raw[:, episode_col]
        else:
            # Không có cột episode -> coi toàn bộ file là 1 chuỗi thời gian liên tục.
            # split_dataset() sẽ dùng block-split thay thế.
            self.episodes = None

        # Chuẩn hóa input LiDAR về [0, 1]
        self.inputs = self.inputs / self.LIDAR_MAX_RANGE

        # Chuẩn hóa target (z-score), lưu lại mean/std để inference dùng ngược lại
        self.target_mean = self.targets.mean(axis=0)
        self.target_std = self.targets.std(axis=0)
        self.target_std[self.target_std < 1e-6] = 1.0  # tránh chia 0
        self.targets_norm = (self.targets - self.target_mean) / self.target_std

        print(f"Loaded dataset from {csv_path}")
        print(f"Total samples: {len(self.inputs)}")
        print(f"Input shape: {self.inputs.shape} | Target shape: {self.targets.shape}")
        print(f"Target mean (v, w): {self.target_mean} | std: {self.target_std}")
        print(f"Episode column detected: {'yes -> ' + header[episode_col] if episode_col is not None else 'no (dùng block-split theo thời gian)'}")

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.targets_norm[idx]


def split_dataset(dataset, val_split=0.2, n_blocks=10, seed=SEED):
    """
    Chia train/val sao cho các sample gần nhau về thời gian không bị xé lẻ
    giữa 2 tập (giảm data leakage do các frame LiDAR liên tiếp gần giống nhau).

    - Nếu có cột episode: chọn ngẫu nhiên (val_split) tỷ lệ episode làm val,
      toàn bộ sample của episode đó đi theo tập val.
    - Nếu không có: chia dữ liệu thành n_blocks khối liên tục theo thời gian,
      rồi chọn ngẫu nhiên (val_split) tỷ lệ khối làm val (blocked split).
    """
    rng = np.random.RandomState(seed)
    n = len(dataset)

    if dataset.episodes is not None:
        unique_eps = np.unique(dataset.episodes)
        rng.shuffle(unique_eps)
        n_val_eps = max(1, int(len(unique_eps) * val_split))
        val_eps = set(unique_eps[:n_val_eps])
        val_idx = np.where(np.isin(dataset.episodes, list(val_eps)))[0]
        train_idx = np.where(~np.isin(dataset.episodes, list(val_eps)))[0]
    else:
        block_edges = np.linspace(0, n, n_blocks + 1).astype(int)
        block_ids = np.arange(n_blocks)
        rng.shuffle(block_ids)
        n_val_blocks = max(1, int(n_blocks * val_split))
        val_blocks = set(block_ids[:n_val_blocks])

        val_mask = np.zeros(n, dtype=bool)
        for b in val_blocks:
            val_mask[block_edges[b]:block_edges[b + 1]] = True
        val_idx = np.where(val_mask)[0]
        train_idx = np.where(~val_mask)[0]

    return Subset(dataset, train_idx.tolist()), Subset(dataset, val_idx.tolist())


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_abs_err = np.zeros(2, dtype=np.float64)
    n = 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            total_loss += loss.item() * inputs.size(0)
            total_abs_err += torch.abs(outputs - targets).sum(dim=0).cpu().numpy()
            n += inputs.size(0)
    return total_loss / n, total_abs_err / n  # avg loss, MAE per output (trong không gian đã chuẩn hóa)


# --- 3. Tiến trình Huấn luyện ---
def train_model(csv_path, save_path, epochs=100, batch_size=64, lr=0.001,
                 val_split=0.2, weight_decay=1e-5, patience=15, dropout=0.1, input_dim=60):
    set_seed()

    dataset = DAggerDataset(csv_path, input_dim=input_dim)
    train_dataset, val_dataset = split_dataset(dataset, val_split=val_split)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DAggerMLP(input_dim=input_dim, output_dim=2, dropout=dropout).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_val_loss = float('inf')
    epochs_no_improve = 0

    print(f"\nTraining on: {device}")
    print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")
    print(f"Epochs: {epochs} | Batch Size: {batch_size} | LR: {lr} | Weight decay: {weight_decay} | Patience: {patience}\n")

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * inputs.size(0)
        train_loss /= len(train_dataset)

        val_loss, val_mae = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_loss)

        # MAE quy đổi lại về đơn vị gốc (m/s, rad/s) để dễ đọc
        val_mae_real = val_mae * dataset.target_std

        if (epoch + 1) % 5 == 0 or epoch == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f} "
                  f"| MAE v: {val_mae_real[0]:.4f} m/s | MAE w: {val_mae_real[1]:.4f} rad/s | LR: {current_lr:.6f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            dir_name = os.path.dirname(save_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            torch.save(model.state_dict(), save_path)

            # Lưu normalization stats cùng với model để inference dùng lại
            norm_path = os.path.splitext(save_path)[0] + '_norm.json'
            with open(norm_path, 'w') as f:
                json.dump({
                    'lidar_max_range': dataset.LIDAR_MAX_RANGE,
                    'target_mean': dataset.target_mean.tolist(),
                    'target_std': dataset.target_std.tolist(),
                }, f, indent=2)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"\nEarly stopping tại epoch {epoch+1} (val loss không cải thiện sau {patience} epoch liên tiếp).")
                break

    print(f"\nTraining Finished! Best Val Loss: {best_val_loss:.6f}")
    print(f"Best model weights saved to: {save_path}")
    print(f"Normalization stats saved to: {os.path.splitext(save_path)[0] + '_norm.json'}")
    print("LƯU Ý: khi inference thực tế, phải nhân output với target_std rồi cộng target_mean "
          "để quy đổi ngược về (linear_v, angular_z) thật.")


if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser(description="Train DAgger Model for F1TENTH")

    parser.add_argument('--csv', type=str,
                        default='final_combined_37500.csv',
                        help='Đường dẫn tới file CSV dataset')
    parser.add_argument('--model', type=str,
                        default='final_combined_37500.pth',
                        help='Đường dẫn lưu file trọng số mô hình (.pth)')
    parser.add_argument('--input_dim', type=int, default=60,
                        help='Số cột LiDAR trong CSV (PHẢI khớp đúng số beam đã thu thập, '
                             'vd 90 cho dagger_dataset_sim_90.csv, 60 cho dataset cũ dagger_dataset_sim_4.csv). '
                             'Sai giá trị này sẽ khiến targets bị đọc lệch sang cột LiDAR khác!')
    parser.add_argument('--epochs', type=int, default=100, help='Số lượng epochs tối đa')
    parser.add_argument('--batch_size', type=int, default=32, help='Batch size')
    parser.add_argument('--lr', type=float, default=0.001, help='Learning rate')
    parser.add_argument('--weight_decay', type=float, default=1e-5, help='Weight decay (L2 regularization)')
    parser.add_argument('--patience', type=int, default=15, help='Early stopping patience (số epoch)')
    parser.add_argument('--dropout', type=float, default=0.1, help='Dropout rate')

    args, unknown = parser.parse_known_args()

    train_model(
        csv_path=args.csv,
        save_path=args.model,
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr=args.lr,
        val_split=0.2,
        weight_decay=args.weight_decay,
        patience=args.patience,
        dropout=args.dropout,
        input_dim=args.input_dim,
    )

Loaded dataset from final_combined_37500.csv
Total samples: 38216
Input shape: (38216, 60) | Target shape: (38216, 2)
Target mean (v, w): [2.4377425  0.12915474] | std: [0.9534305 0.1454656]
Episode column detected: no (dùng block-split theo thời gian)

Training on: cuda
Train samples: 30572 | Val samples: 7644
Epochs: 100 | Batch Size: 32 | LR: 0.001 | Weight decay: 1e-05 | Patience: 15

Epoch [1/100] | Train Loss: 0.703849 | Val Loss: 0.626829 | MAE v: 0.4361 m/s | MAE w: 0.0740 rad/s | LR: 0.001000
Epoch [5/100] | Train Loss: 0.440780 | Val Loss: 0.428662 | MAE v: 0.2636 m/s | MAE w: 0.0627 rad/s | LR: 0.001000
Epoch [10/100] | Train Loss: 0.349271 | Val Loss: 0.343222 | MAE v: 0.2122 m/s | MAE w: 0.0576 rad/s | LR: 0.001000
Epoch [15/100] | Train Loss: 0.310591 | Val Loss: 0.301915 | MAE v: 0.1837 m/s | MAE w: 0.0552 rad/s | LR: 0.001000
Epoch [20/100] | Train Loss: 0.288097 | Val Loss: 0.293229 | MAE v: 0.1941 m/s | MAE w: 0.0546 rad/s | LR: 0.001000
Epoch [25/100] | Train Loss: 0

In [ ]:
!pip install onnxscript onnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 16.9 MB/s eta 0:00:00


In [ ]:
#!/usr/bin/env python3
"""
export_onnx.py
───────────────
Export model DAgger (.pth) đã train bằng train.py sang ONNX để chạy trên Jetson
(TensorRT / onnxruntime).

QUAN TRỌNG — khác với script export cũ:
  1. Kiến trúc DAggerMLP giờ có thêm Dropout (so khớp với train.py mới) — nếu không
     khớp kiến trúc, load_state_dict sẽ lệch key và load SAI trọng số (silent bug),
     vì vị trí index của các Linear layer trong nn.Sequential bị đẩy lùi do Dropout
     chen vào giữa.
  2. train.py mới CHUẨN HÓA input (chia 10.0) và target (z-score) trước khi train,
     nên model .pth chỉ nhận input đã chia 10 và trả ra output đã chuẩn hóa —
     KHÔNG phải (linear_v, angular_z) thật.
     -> Script này bọc model gốc trong một wrapper (`DeployWrapper`) để bake luôn
        bước chuẩn hóa input và giải chuẩn hóa (denormalize) output VÀO TRONG đồ thị
        ONNX. Nhờ vậy, phía deploy (C++/Jetson) chỉ cần đưa LiDAR thô (mét) vào và
        nhận thẳng (linear_v, angular_z) thật ra, không cần biết gì về mean/std.
"""

import os
import json
import argparse

import torch
import torch.nn as nn


# --- 1. Kiến trúc model — PHẢI khớp 100% với train.py đã dùng để tạo ra file .pth ---
class DAggerMLP(nn.Module):
    def __init__(self, input_dim=60, output_dim=2, dropout=0.1):
        super(DAggerMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim)
        )

    def forward(self, x):
        return self.network(x)


# --- 2. Wrapper bake normalization/denormalization vào đồ thị export ---
class DeployWrapper(nn.Module):
    """
    Input:  LiDAR thô (mét), shape (batch, input_dim)
    Output: (linear_v, angular_z) thật, shape (batch, 2)
    """
    def __init__(self, base_model, lidar_max_range, target_mean, target_std):
        super().__init__()
        self.base_model = base_model
        self.register_buffer('lidar_max_range', torch.tensor(float(lidar_max_range)))
        self.register_buffer('target_mean', torch.tensor(target_mean, dtype=torch.float32))
        self.register_buffer('target_std', torch.tensor(target_std, dtype=torch.float32))

    def forward(self, x):
        x_norm = x / self.lidar_max_range
        out_norm = self.base_model(x_norm)
        out_real = out_norm * self.target_std + self.target_mean
        return out_real


def export(model_path, onnx_path, norm_path=None, input_dim=60, opset=15):
    if norm_path is None:
        norm_path = os.path.splitext(model_path)[0] + '_norm.json'

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Không tìm thấy file model: {model_path}")
    if not os.path.exists(norm_path):
        raise FileNotFoundError(
            f"Không tìm thấy file normalization stats: {norm_path}\n"
            f"File này được train.py tự tạo ra cùng lúc với .pth. Nếu bạn dùng model "
            f"train bằng phiên bản train.py CŨ (không chuẩn hóa target) thì hãy dùng "
            f"lại script export ONNX bản gốc thay vì script này."
        )

    with open(norm_path, 'r') as f:
        norm_stats = json.load(f)

    lidar_max_range = norm_stats['lidar_max_range']
    target_mean = norm_stats['target_mean']
    target_std = norm_stats['target_std']

    print(f"Loaded normalization stats từ: {norm_path}")
    print(f"  lidar_max_range = {lidar_max_range}")
    print(f"  target_mean (v, w) = {target_mean}")
    print(f"  target_std  (v, w) = {target_std}")

    base_model = DAggerMLP(input_dim=input_dim, output_dim=2)
    state_dict = torch.load(model_path, map_location="cpu")
    missing, unexpected = base_model.load_state_dict(state_dict, strict=True)
    base_model.eval()

    deploy_model = DeployWrapper(base_model, lidar_max_range, target_mean, target_std)
    deploy_model.eval()

    dummy_input = torch.rand(1, input_dim) * lidar_max_range  # mô phỏng LiDAR thô trong khoảng [0, max_range]

    torch.onnx.export(
        deploy_model,
        dummy_input,
        onnx_path,
        opset_version=opset,
        input_names=['lidar_raw'],
        output_names=['cmd_vel'],  # [linear_v, angular_z] thật, đã denormalize
        dynamic_axes={'lidar_raw': {0: 'batch_size'}, 'cmd_vel': {0: 'batch_size'}}
    )
    print(f"\nĐã export ONNX -> {onnx_path}")
    print("Input 'lidar_raw': LiDAR thô tính bằng MÉT, không cần tự chia /10 nữa.")
    print("Output 'cmd_vel': [linear_v, angular_z] THẬT, không cần tự denormalize nữa.")

    # --- 3. Kiểm tra sanity: so sánh output PyTorch vs ONNX (nếu có onnxruntime) ---
    try:
        import onnxruntime as ort
        import numpy as np

        with torch.no_grad():
            torch_out = deploy_model(dummy_input).numpy()

        sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
        onnx_out = sess.run(None, {'lidar_raw': dummy_input.numpy()})[0]

        max_diff = np.abs(torch_out - onnx_out).max()
        print(f"\n[Sanity check] Max diff PyTorch vs ONNX trên dummy input: {max_diff:.8f}")
        if max_diff > 1e-4:
            print("CẢNH BÁO: sai lệch lớn hơn ngưỡng thông thường, kiểm tra lại opset/kiến trúc.")
    except ImportError:
        print("\n(Bỏ qua sanity check vì chưa cài onnxruntime: pip install onnxruntime)")


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description="Export DAgger PyTorch model sang ONNX cho Jetson")
    parser.add_argument('--model', type=str, default='final_combined_37500.pth',
                        help='Đường dẫn file trọng số .pth')
    parser.add_argument('--norm', type=str, default=None,
                        help="Đường dẫn file *_norm.json (mặc định: suy ra từ --model)")
    parser.add_argument('--onnx', type=str, default='final_combined_37500.onnx',
                        help='Đường dẫn file ONNX xuất ra')
    parser.add_argument('--input_dim', type=int, default=60,
                        help='Số chiều input (số beam LiDAR). PHẢI khớp với input_dim đã dùng lúc train.py '
                             '(90 cho dagger_dataset_sim_90.csv, 60 cho dataset cũ)')
    parser.add_argument('--opset', type=int, default=15, help='ONNX opset version')

    args, unknown = parser.parse_known_args()

    export(
        model_path=args.model,
        onnx_path=args.onnx,
        norm_path=args.norm,
        input_dim=args.input_dim,
        opset=args.opset,
    )

Loaded normalization stats từ: final_combined_37500_norm.json
  lidar_max_range = 10.0
  target_mean (v, w) = [2.4377424716949463, 0.1291547417640686]
  target_std  (v, w) = [0.9534304738044739, 0.14546559751033783]


/tmp/ipykernel_1125/3539655425.py:106: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0730 10:09:11.391000 1125 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 15 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DeployWrapper([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DeployWrapper([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅

Đã export ONNX -> final_combined_37500.onnx
Input 'lidar_raw': LiDAR thô tính bằng MÉT, không cần tự chia /10 nữa.
Output 'cmd_vel': [linear_v, angular_z] THẬT, không cần tự denormalize nữa.

(Bỏ qua sanity check vì chưa cài onnxruntime: pip install onnxruntime)


In [ ]:
import onnx
model = onnx.load("final_combined_37500.onnx")
model.ir_version = 9  # Ép định dạng file về bản tương thích với Docker
onnx.save(model, "final_combined_37500.onnx")
